<a href="https://colab.research.google.com/github/Tipusultan199/kkk/blob/main/Phase2B_Colab_LabelOnly_Debug_READY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2B — Column Guard + Active-Only Export Debug Notebook

This notebook is organized for Google Colab/professor sharing. It keeps the original Phase 2B logic: validate label CSV schema, reorder columns to the canonical training schema, and export only active/action rows for non-T0 trials.

For this uploaded sample, the strict original guard finds extra ET columns and would skip the file. A clearly marked Colab-safe repair cell drops only unsupported extra columns so the active-only export can still be inspected.

In [1]:
# ============================================================
# CELL 1 — Import libraries and define debug helper
# This cell loads the required Python packages and creates a pass/fail printer.
# ============================================================

from __future__ import annotations

import csv
import glob
import re
import shutil
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

def print_step_result(step_name, passed, details=""):
    """Print a clear pass/fail message after each Phase 2B step."""
    if passed:
        print(f"✅ CLEANING APPLIED CORRECTLY — {step_name}")
    else:
        print(f"❌ CLEANING NOT APPLIED CORRECTLY — {step_name}")
    if details:
        print(details)


In [3]:
# ============================================================
# CELL 2 — Upload HSMM-labeled CSV and set output folder
# This cell lets you upload the Phase 2A HSMM-labeled CSV directly in Colab.
# Example input: 063_T24_HSMM.csv
# ============================================================

from pathlib import Path
from google.colab import files

print("Please upload your HSMM-labeled CSV file, for example: 063_T24_HSMM.csv")

uploaded = files.upload()

if len(uploaded) == 0:
    raise FileNotFoundError("No CSV file was uploaded.")

# Use the first uploaded file
uploaded_filename = list(uploaded.keys())[0]
INPUT_CSV = Path("/content") / uploaded_filename

OUTPUT_DIR = Path("/content/phase2B_debug_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STRICT_ORIGINAL_MODE = False
# True  = behave exactly like original Phase 2B and skip if extras/missing columns exist.
# False = show the original strict failure, then drop only unsupported extra columns for Colab inspection/export.

print("Uploaded file:", uploaded_filename)
print("INPUT_CSV    :", INPUT_CSV)
print("Exists       :", INPUT_CSV.exists())
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("STRICT_ORIGINAL_MODE:", STRICT_ORIGINAL_MODE)

print_step_result("CELL 2 — CSV uploaded and paths configured", INPUT_CSV.exists())

Please upload your HSMM-labeled CSV file, for example: 063_T24_HSMM.csv


Saving 063_T24_HSMM.csv to 063_T24_HSMM.csv
Uploaded file: 063_T24_HSMM.csv
INPUT_CSV    : /content/063_T24_HSMM.csv
Exists       : True
OUTPUT_DIR   : /content/phase2B_debug_outputs
STRICT_ORIGINAL_MODE: False
✅ CLEANING APPLIED CORRECTLY — CELL 2 — CSV uploaded and paths configured


In [4]:
# ============================================================
# CELL 3 — Original Phase 2B configuration and helper functions
# This cell keeps the main code logic from your original Phase-2-B notebook.
# ============================================================

# Canonical header order from original Phase-2-B notebook
CANONICAL_BASE = [
    "Timestamp_seconds",
    # optional Timestamp_ms will be inserted after Timestamp_seconds if present
    "EMG_Ch1","EMG_Ch2","EMG_Ch3","EMG_Ch4",
    "EEG_Ch1","EEG_Ch2","EEG_Ch3","EEG_Ch4","EEG_Ch5","EEG_Ch6","EEG_Ch7","EEG_Ch8",
    "ET_GazeLeftx","ET_GazeLefty","ET_GazeRightx","ET_GazeRighty",
    "ET_PupilLeft","ET_PupilRight",
    "ET_ValidityLeftEye","ET_ValidityRightEye",
    "ET_Blink","ET_Fixation","ET_Worn",
    # optional ET_DistanceLeft/ET_DistanceRight will be inserted after ET_Worn if present
    "ET_GyroX","ET_GyroY","ET_GyroZ",
    "ET_AccX","ET_AccY","ET_AccZ",
    "ET_HeadRotationPitch","ET_HeadRotationYaw","ET_HeadRotationRoll",
    "active_raw","active","active_prob","label_action",
    "subject_id","task","trial","label_11","task_target",
]

OPTIONAL_COLS = {
    "Timestamp_ms",
    "ET_DistanceLeft","ET_DistanceRight",
}

ALLOWED_SET = set(CANONICAL_BASE) | OPTIONAL_COLS
REQUIRED_SET = set(CANONICAL_BASE)  # optionals not required

def _clean_header(header):
    out = []
    for i, h in enumerate(header):
        h = (h or "")
        if i == 0:
            h = h.replace("\ufeff", "")
        out.append(h.strip())
    return out

def _target_order_for(header: list[str]) -> list[str]:
    has_ts_ms = "Timestamp_ms" in header
    has_distL = "ET_DistanceLeft" in header
    has_distR = "ET_DistanceRight" in header

    target = []
    for name in CANONICAL_BASE:
        if name == "Timestamp_seconds":
            target.append(name)
            if has_ts_ms:
                target.append("Timestamp_ms")
        elif name == "ET_Worn":
            target.append(name)
            if has_distL:
                target.append("ET_DistanceLeft")
            if has_distR:
                target.append("ET_DistanceRight")
        else:
            target.append(name)

    # Only keep those that actually exist in the file
    return [c for c in target if c in header]

def _is_task0_file(path: Path) -> bool:
    """
    Detect Task 0 from filename using the same convention as the labeler:
        - (T|M)(digits)
        - FIRST digit after T/M = task
        - remaining digits = trial
    """
    stem = path.stem
    m = re.search(r'(?:^|_)(T|M)(\d{2,})', stem, flags=re.I)
    if not m:
        return False

    kind = m.group(1).upper()
    if kind != "T":
        return False

    digits = m.group(2)
    task = int(digits[0])
    return task == 0

print("Required canonical columns:", len(CANONICAL_BASE))
print("Optional columns:", sorted(OPTIONAL_COLS))
print_step_result("CELL 3 — Original Phase 2B helpers loaded", True)


Required canonical columns: 42
Optional columns: ['ET_DistanceLeft', 'ET_DistanceRight', 'Timestamp_ms']
✅ CLEANING APPLIED CORRECTLY — CELL 3 — Original Phase 2B helpers loaded


In [5]:
# ============================================================
# CELL 4 — Load HSMM-labeled CSV and inspect raw shape/columns
# This cell reads the Phase 2A/HSMM output before Phase 2B filtering.
# ============================================================

df_raw = pd.read_csv(INPUT_CSV, low_memory=False)
header_raw = list(df_raw.columns)
header = _clean_header(header_raw)

print("Raw shape:", df_raw.shape)
print("\nRaw columns:")
for i, c in enumerate(header):
    print(f"{i+1:02d}. {c}")

print("\nFirst 5 rows:")
display(df_raw.head())

passed = len(df_raw) > 0 and "active" in df_raw.columns
print_step_result("CELL 4 — Raw HSMM CSV loaded", passed)


Raw shape: (2097, 58)

Raw columns:
01. Timestamp_seconds
02. Timestamp_ms
03. EMG_Ch1
04. EMG_Ch2
05. EMG_Ch3
06. EMG_Ch4
07. EEG_Ch1
08. EEG_Ch2
09. EEG_Ch3
10. EEG_Ch4
11. EEG_Ch5
12. EEG_Ch6
13. EEG_Ch7
14. EEG_Ch8
15. ET_TimeSignal
16. ET_PupilLeft
17. ET_PupilRight
18. ET_DistanceLeft
19. ET_DistanceRight
20. ET_GazeLeftx
21. ET_GazeLefty
22. ET_GazeRightx
23. ET_GazeRighty
24. ET_ValidityLeftEye
25. ET_ValidityRightEye
26. ET_GyroX
27. ET_GyroY
28. ET_GyroZ
29. ET_AccX
30. ET_AccY
31. ET_AccZ
32. ET_HeadRotationPitch
33. ET_HeadRotationYaw
34. ET_HeadRotationRoll
35. ET_Blink
36. ET_Gaze3dEyeballXLeft
37. ET_Gaze3dEyeballYLeft
38. ET_Gaze3dEyeballZLeft
39. ET_Gaze3dEyeballXRight
40. ET_Gaze3dEyeballYRight
41. ET_Gaze3dEyeballZRight
42. ET_Gaze3dOpticalAxisXLeft
43. ET_Gaze3dOpticalAxisYLeft
44. ET_Gaze3dOpticalAxisZLeft
45. ET_Gaze3dOpticalAxisXRight
46. ET_Gaze3dOpticalAxisYRight
47. ET_Gaze3dOpticalAxisZRight
48. ET_Fixation
49. ET_Worn
50. active_raw
51. active
52. active_pro

,Timestamp_seconds,Timestamp_ms,EMG_Ch1,EMG_Ch2,EMG_Ch3,EMG_Ch4,EEG_Ch1,EEG_Ch2,EEG_Ch3,EEG_Ch4,EEG_Ch5,EEG_Ch6,EEG_Ch7,EEG_Ch8,ET_TimeSignal,ET_PupilLeft,ET_PupilRight,ET_DistanceLeft,ET_DistanceRight,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_ValidityLeftEye,ET_ValidityRightEye,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll,ET_Blink,ET_Gaze3dEyeballXLeft,ET_Gaze3dEyeballYLeft,ET_Gaze3dEyeballZLeft,ET_Gaze3dEyeballXRight,ET_Gaze3dEyeballYRight,ET_Gaze3dEyeballZRight,ET_Gaze3dOpticalAxisXLeft,ET_Gaze3dOpticalAxisYLeft,ET_Gaze3dOpticalAxisZLeft,ET_Gaze3dOpticalAxisXRight,ET_Gaze3dOpticalAxisYRight,ET_Gaze3dOpticalAxisZRight,ET_Fixation,ET_Worn,active_raw,active,active_prob,label_action,subject_id,task,trial,label_11,task_target
0,0.000,0.0,-126.2631,-1.5754,36.9821,-42.3301,104424.4413,97475.1924,104742.5813,117912.4794,134633.4386,61568.6767,62149.6021,45517.7551,2.0,3.6214,3.5148,-1.0,-1.0,0.4706,0.4675,0.4706,0.4675,0.0,0.0,-0.2169,-0.0365,0.8684,-0.7996,-0.7262,9.8256,85.2129,-1.9398,11.6523,0.0,-30.25,6.25,-34.3438,32.6875,9.4062,-35.4375,0.1132,0.0776,0.9905,-0.1703,0.0522,0.984,0.0,0.0,0,0,0.0,0,10,2,4,0,0
1,0.004,4.0,46.4246,-1.2888,-15.5247,-27.5305,104462.7208,97482.0962,104770.1333,117997.9444,134703.4319,61589.8971,62268.6444,45581.1649,2.0,3.6214,3.5148,-1.0,-1.0,0.4706,0.4675,0.4706,0.4675,0.0,0.0,-0.2169,-0.0365,0.8684,-0.7996,-0.7262,9.8256,85.2129,-1.9398,11.6523,0.0,-30.25,6.25,-34.3438,32.6875,9.4062,-35.4375,0.1132,0.0776,0.9905,-0.1703,0.0522,0.984,0.0,0.0,0,0,0.0,0,10,2,4,0,0
2,0.008,8.0,106.2493,-11.2270,-9.0031,-28.8864,104483.3394,97487.8389,104794.9249,118028.2891,134736.9504,61603.7531,62333.2962,45634.4262,2.0,3.6214,3.5148,-1.0,-1.0,0.4706,0.4675,0.4706,0.4675,0.0,0.0,-0.2169,-0.0365,0.8684,-0.7996,-0.7262,9.8256,85.2129,-1.9398,11.6523,0.0,-30.25,6.25,-34.3438,32.6875,9.4062,-35.4375,0.1132,0.0776,0.9905,-0.1703,0.0522,0.984,0.0,0.0,0,0,0.0,0,10,2,4,0,0
3,0.012,12.0,-20.8769,1.2063,46.7704,-36.4677,104463.5218,97486.5396,104784.0513,117974.2510,134696.9280,61594.7976,62268.2234,45611.4722,2.0,3.6214,3.5148,-1.0,-1.0,0.4706,0.4675,0.4706,0.4675,0.0,0.0,-0.2169,-0.0365,0.8684,-0.7996,-0.7262,9.8256,85.2129,-1.9398,11.6523,0.0,-30.25,6.25,-34.3438,32.6875,9.4062,-35.4375,0.1132,0.0776,0.9905,-0.1703,0.0522,0.984,0.0,0.0,0,0,0.0,0,10,2,4,0,0
4,0.016,16.0,-186.0476,39.1754,85.3186,-27.0042,104442.0450,97482.1887,104755.9037,117942.2217,134661.4083,61581.1145,62199.3781,45557.7392,2.0,3.6214,3.5148,-1.0,-1.0,0.4706,0.4675,0.4706,0.4675,0.0,0.0,-0.2169,-0.0365,0.8684,-0.7996,-0.7262,9.8256,85.2129,-1.9398,11.6523,0.0,-30.25,6.25,-34.3438,32.6875,9.4062,-35.4375,0.1132,0.0776,0.9905,-0.1703,0.0522,0.984,0.0,0.0,0,0,0.0,0,10,2,4,0,0


✅ CLEANING APPLIED CORRECTLY — CELL 4 — Raw HSMM CSV loaded


In [7]:
# ============================================================
# CELL 5 — Original strict Phase 2B column guard check
# This cell checks missing required columns and extra unsupported columns.
# Original Phase 2B would skip the file if either list is non-empty.
# In this Colab debug notebook, this is a diagnostic check.
# ============================================================

set_cur = set(header)
extras = sorted(list(set_cur - ALLOWED_SET))
missing_required = sorted(list(REQUIRED_SET - set_cur))

print("Missing required columns:", missing_required if missing_required else "None")
print("Extra unsupported columns:", extras if extras else "None")
print("Number of extra columns:", len(extras))
print("Number of missing required columns:", len(missing_required))

strict_guard_passed = (len(extras) == 0 and len(missing_required) == 0)

if strict_guard_passed:
    print("\n✅ Original Phase 2B decision: EXPORT is allowed.")
    print_step_result("CELL 5 — Original strict column guard check", True)

else:
    print("\n⚠️ Original Phase 2B decision: this file would be SKIPPED.")
    print("Reason: original code treats extra/missing columns as invalid schema.")

    if len(missing_required) == 0 and len(extras) > 0:
        print("\nInterpretation:")
        print("The required label columns are present.")
        print("The only issue is extra unsupported columns.")
        print("This is safe to repair in Colab by dropping only the extra columns.")
        print("Continue to the next cell if STRICT_ORIGINAL_MODE=False.")
        print("\n⚠️ STRICT ORIGINAL CHECK FAILED, BUT REPAIR MODE CAN CONTINUE.")
    else:
        print("\n❌ Required columns are missing. This is a real problem.")

if (not strict_guard_passed) and STRICT_ORIGINAL_MODE:
    raise ValueError("STRICT_ORIGINAL_MODE=True: stopping because original Phase 2B would skip this file.")

Missing required columns: None
Extra unsupported columns: ['ET_Gaze3dEyeballXLeft', 'ET_Gaze3dEyeballXRight', 'ET_Gaze3dEyeballYLeft', 'ET_Gaze3dEyeballYRight', 'ET_Gaze3dEyeballZLeft', 'ET_Gaze3dEyeballZRight', 'ET_Gaze3dOpticalAxisXLeft', 'ET_Gaze3dOpticalAxisXRight', 'ET_Gaze3dOpticalAxisYLeft', 'ET_Gaze3dOpticalAxisYRight', 'ET_Gaze3dOpticalAxisZLeft', 'ET_Gaze3dOpticalAxisZRight', 'ET_TimeSignal']
Number of extra columns: 13
Number of missing required columns: 0

⚠️ Original Phase 2B decision: this file would be SKIPPED.
Reason: original code treats extra/missing columns as invalid schema.

Interpretation:
The required label columns are present.
The only issue is extra unsupported columns.
This is safe to repair in Colab by dropping only the extra columns.
Continue to the next cell if STRICT_ORIGINAL_MODE=False.

⚠️ STRICT ORIGINAL CHECK FAILED, BUT REPAIR MODE CAN CONTINUE.


In [8]:
# ============================================================
# CELL 6 — Colab-safe schema repair for this single-file debug
# This cell keeps only canonical + optional columns and drops unsupported extras.
# This does not change labels or signal values; it only removes columns Phase 2B does not allow.
# ============================================================

target_order = _target_order_for(header)

print("Target canonical order length:", len(target_order))
print("Target columns:")
for i, c in enumerate(target_order):
    print(f"{i+1:02d}. {c}")

if missing_required:
    print("\nCannot repair missing required columns safely.")
    df_schema = None
    passed = False
else:
    df_schema = df_raw[target_order].copy()
    dropped_cols = [c for c in header if c not in target_order]
    print("\nDropped unsupported columns:")
    print(dropped_cols if dropped_cols else "None")
    print("\nShape before schema repair:", df_raw.shape)
    print("Shape after schema repair :", df_schema.shape)
    print("Rows changed:", len(df_raw) != len(df_schema))
    print("Columns removed:", df_raw.shape[1] - df_schema.shape[1])
    passed = (
        len(df_schema) == len(df_raw)
        and list(df_schema.columns) == target_order
        and not any(c in df_schema.columns for c in extras)
    )

print("\nPreview after schema repair/reorder:")
display(df_schema.head() if df_schema is not None else pd.DataFrame())

print_step_result("CELL 6 — Canonical schema repair/reorder", passed)


Target canonical order length: 45
Target columns:
01. Timestamp_seconds
02. Timestamp_ms
03. EMG_Ch1
04. EMG_Ch2
05. EMG_Ch3
06. EMG_Ch4
07. EEG_Ch1
08. EEG_Ch2
09. EEG_Ch3
10. EEG_Ch4
11. EEG_Ch5
12. EEG_Ch6
13. EEG_Ch7
14. EEG_Ch8
15. ET_GazeLeftx
16. ET_GazeLefty
17. ET_GazeRightx
18. ET_GazeRighty
19. ET_PupilLeft
20. ET_PupilRight
21. ET_ValidityLeftEye
22. ET_ValidityRightEye
23. ET_Blink
24. ET_Fixation
25. ET_Worn
26. ET_DistanceLeft
27. ET_DistanceRight
28. ET_GyroX
29. ET_GyroY
30. ET_GyroZ
31. ET_AccX
32. ET_AccY
33. ET_AccZ
34. ET_HeadRotationPitch
35. ET_HeadRotationYaw
36. ET_HeadRotationRoll
37. active_raw
38. active
39. active_prob
40. label_action
41. subject_id
42. task
43. trial
44. label_11
45. task_target

Dropped unsupported columns:
['ET_TimeSignal', 'ET_Gaze3dEyeballXLeft', 'ET_Gaze3dEyeballYLeft', 'ET_Gaze3dEyeballZLeft', 'ET_Gaze3dEyeballXRight', 'ET_Gaze3dEyeballYRight', 'ET_Gaze3dEyeballZRight', 'ET_Gaze3dOpticalAxisXLeft', 'ET_Gaze3dOpticalAxisYLeft', 'ET_G

,Timestamp_seconds,Timestamp_ms,EMG_Ch1,EMG_Ch2,EMG_Ch3,EMG_Ch4,EEG_Ch1,EEG_Ch2,EEG_Ch3,EEG_Ch4,EEG_Ch5,EEG_Ch6,EEG_Ch7,EEG_Ch8,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_PupilLeft,ET_PupilRight,ET_ValidityLeftEye,ET_ValidityRightEye,ET_Blink,ET_Fixation,ET_Worn,ET_DistanceLeft,ET_DistanceRight,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll,active_raw,active,active_prob,label_action,subject_id,task,trial,label_11,task_target
0,0.000,0.0,-126.2631,-1.5754,36.9821,-42.3301,104424.4413,97475.1924,104742.5813,117912.4794,134633.4386,61568.6767,62149.6021,45517.7551,0.4706,0.4675,0.4706,0.4675,3.6214,3.5148,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-0.2169,-0.0365,0.8684,-0.7996,-0.7262,9.8256,85.2129,-1.9398,11.6523,0,0,0.0,0,10,2,4,0,0
1,0.004,4.0,46.4246,-1.2888,-15.5247,-27.5305,104462.7208,97482.0962,104770.1333,117997.9444,134703.4319,61589.8971,62268.6444,45581.1649,0.4706,0.4675,0.4706,0.4675,3.6214,3.5148,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-0.2169,-0.0365,0.8684,-0.7996,-0.7262,9.8256,85.2129,-1.9398,11.6523,0,0,0.0,0,10,2,4,0,0
2,0.008,8.0,106.2493,-11.2270,-9.0031,-28.8864,104483.3394,97487.8389,104794.9249,118028.2891,134736.9504,61603.7531,62333.2962,45634.4262,0.4706,0.4675,0.4706,0.4675,3.6214,3.5148,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-0.2169,-0.0365,0.8684,-0.7996,-0.7262,9.8256,85.2129,-1.9398,11.6523,0,0,0.0,0,10,2,4,0,0
3,0.012,12.0,-20.8769,1.2063,46.7704,-36.4677,104463.5218,97486.5396,104784.0513,117974.2510,134696.9280,61594.7976,62268.2234,45611.4722,0.4706,0.4675,0.4706,0.4675,3.6214,3.5148,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-0.2169,-0.0365,0.8684,-0.7996,-0.7262,9.8256,85.2129,-1.9398,11.6523,0,0,0.0,0,10,2,4,0,0
4,0.016,16.0,-186.0476,39.1754,85.3186,-27.0042,104442.0450,97482.1887,104755.9037,117942.2217,134661.4083,61581.1145,62199.3781,45557.7392,0.4706,0.4675,0.4706,0.4675,3.6214,3.5148,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-0.2169,-0.0365,0.8684,-0.7996,-0.7262,9.8256,85.2129,-1.9398,11.6523,0,0,0.0,0,10,2,4,0,0


✅ CLEANING APPLIED CORRECTLY — CELL 6 — Canonical schema repair/reorder


In [9]:
# ============================================================
# CELL 7 — Check active labels before active-only filtering
# This cell prints active/rest counts before removing active==0 rows.
# ============================================================

required_filter_cols = ["active", "label_action", "label_11", "task_target", "subject_id", "task", "trial"]

print("Required filter/metadata columns:")
for c in required_filter_cols:
    print(f"{c}: {'FOUND' if c in df_schema.columns else 'MISSING'}")

print("\nCounts before filtering:")
for c in ["active", "label_action", "label_11", "task_target", "subject_id", "task", "trial"]:
    if c in df_schema.columns:
        print(f"\n{c}:")
        print(df_schema[c].value_counts(dropna=False).sort_index())

is_task0 = _is_task0_file(INPUT_CSV)
print("\nDetected T0/REST trial from filename:", is_task0)

passed = all(c in df_schema.columns for c in required_filter_cols)
print_step_result("CELL 7 — Active labels available before filter", passed)


Required filter/metadata columns:
active: FOUND
label_action: FOUND
label_11: FOUND
task_target: FOUND
subject_id: FOUND
task: FOUND
trial: FOUND

Counts before filtering:

active:
active
0    1000
1    1097
Name: count, dtype: int64

label_action:
label_action
0    1000
1    1097
Name: count, dtype: int64

label_11:
label_11
0    1000
2    1097
Name: count, dtype: int64

task_target:
task_target
0    1000
2    1097
Name: count, dtype: int64

subject_id:
subject_id
10    2097
Name: count, dtype: int64

task:
task
2    2097
Name: count, dtype: int64

trial:
trial
4    2097
Name: count, dtype: int64

Detected T0/REST trial from filename: False
✅ CLEANING APPLIED CORRECTLY — CELL 7 — Active labels available before filter


In [10]:
# ============================================================
# CELL 8 — Apply original Phase 2B active-only export logic
# This cell keeps all rows for T0 trials; otherwise it keeps only active==1 rows.
# ============================================================

df_filter_input = df_schema.copy()
before_rows = len(df_filter_input)

# Convert active to integer safely before filtering, matching the original intent.
df_filter_input["active"] = pd.to_numeric(df_filter_input["active"], errors="coerce").fillna(0).astype("int64")

if is_task0:
    df_labelonly = df_filter_input.copy()
    rows_removed = 0
    filter_mode = "T0 passthrough — no active-only filtering"
else:
    df_labelonly = df_filter_input[df_filter_input["active"].astype("int64") == 1].copy()
    rows_removed = before_rows - len(df_labelonly)
    filter_mode = "Non-T0 active-only filtering"

print("Filter mode:", filter_mode)
print("Rows before filter:", before_rows)
print("Rows removed active==0:", rows_removed)
print("Rows after filter:", len(df_labelonly))

print("\nCounts after filtering:")
for c in ["active", "label_action", "label_11", "task_target", "subject_id", "task", "trial"]:
    if c in df_labelonly.columns:
        print(f"\n{c}:")
        print(df_labelonly[c].value_counts(dropna=False).sort_index())

print("\nPreview after active-only filtering:")
display(df_labelonly.head())
display(df_labelonly.tail())

if is_task0:
    passed = len(df_labelonly) == before_rows
else:
    passed = len(df_labelonly) > 0 and (df_labelonly["active"].astype("int64") == 1).all()

print_step_result("CELL 8 — Active-only filtering applied", passed)


Filter mode: Non-T0 active-only filtering
Rows before filter: 2097
Rows removed active==0: 1000
Rows after filter: 1097

Counts after filtering:

active:
active
1    1097
Name: count, dtype: int64

label_action:
label_action
1    1097
Name: count, dtype: int64

label_11:
label_11
2    1097
Name: count, dtype: int64

task_target:
task_target
2    1097
Name: count, dtype: int64

subject_id:
subject_id
10    1097
Name: count, dtype: int64

task:
task
2    1097
Name: count, dtype: int64

trial:
trial
4    1097
Name: count, dtype: int64

Preview after active-only filtering:


,Timestamp_seconds,Timestamp_ms,EMG_Ch1,EMG_Ch2,EMG_Ch3,EMG_Ch4,EEG_Ch1,EEG_Ch2,EEG_Ch3,EEG_Ch4,EEG_Ch5,EEG_Ch6,EEG_Ch7,EEG_Ch8,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_PupilLeft,ET_PupilRight,ET_ValidityLeftEye,ET_ValidityRightEye,ET_Blink,ET_Fixation,ET_Worn,ET_DistanceLeft,ET_DistanceRight,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll,active_raw,active,active_prob,label_action,subject_id,task,trial,label_11,task_target
999,3.996,3996.0,-35.6730,7.5761,19.9850,108.1464,104412.1595,97503.4998,104708.9312,117952.2484,134692.5810,61608.7579,62264.5347,45589.4857,0.6024,0.6104,0.6024,0.6104,4.3070,4.3687,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-26.9252,2.1304,-12.4931,-0.5567,-4.5976,8.9107,61.3502,-42.3269,9.3288,1,1,0.377244,1,10,2,4,2,2
1000,4.000,4000.0,78.8217,-149.2976,-220.4043,15.8415,104415.8378,97510.5967,104693.6469,117970.2568,134702.0429,61611.5667,62264.0619,45562.3994,0.5972,0.6103,0.5972,0.6103,4.3575,4.3737,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-26.4346,1.6322,-12.2411,-0.5483,-4.6155,8.8952,61.2693,-42.3602,9.3155,1,1,0.377244,1,10,2,4,2,2
1001,4.004,4004.0,232.5416,-27.1953,-131.1567,-62.6811,104445.4114,97525.7896,104729.5683,118029.8235,134762.6485,61628.4776,62391.4939,45650.5991,0.5954,0.6110,0.5954,0.6110,4.3721,4.3964,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-25.9348,1.1527,-11.9698,-0.5526,-4.6271,8.8649,61.1919,-42.4009,9.3027,1,1,0.377244,1,10,2,4,2,2
1002,4.008,4008.0,164.6698,65.0828,50.5351,-30.8734,104448.5902,97529.2024,104754.3236,118023.2275,134765.1745,61627.1919,62412.2693,45693.8504,0.5948,0.6100,0.5948,0.6100,4.3721,4.4520,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-25.3309,0.5826,-11.5988,-0.5776,-4.6320,8.8112,61.1000,-42.4330,9.2862,1,1,0.377244,1,10,2,4,2,2
1003,4.012,4012.0,-95.5774,-39.0934,154.8412,82.2288,104420.3970,97515.3359,104725.5367,117970.0322,134710.8885,61607.8690,62302.1497,45616.2374,0.5938,0.6088,0.5938,0.6088,4.3819,4.4500,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-24.9359,0.1354,-11.3474,-0.5924,-4.6352,8.7764,61.0380,-42.4454,9.2732,1,1,0.377244,1,10,2,4,2,2


,Timestamp_seconds,Timestamp_ms,EMG_Ch1,EMG_Ch2,EMG_Ch3,EMG_Ch4,EEG_Ch1,EEG_Ch2,EEG_Ch3,EEG_Ch4,EEG_Ch5,EEG_Ch6,EEG_Ch7,EEG_Ch8,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_PupilLeft,ET_PupilRight,ET_ValidityLeftEye,ET_ValidityRightEye,ET_Blink,ET_Fixation,ET_Worn,ET_DistanceLeft,ET_DistanceRight,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll,active_raw,active,active_prob,label_action,subject_id,task,trial,label_11,task_target
2091,8.364,8364.0,-39.4391,-121.6410,-282.8916,100.2841,104557.8257,97677.0598,104793.3578,118005.4055,134804.3440,61716.3508,62387.1685,45611.6916,0.4475,0.4541,0.4475,0.4541,4.2224,4.2370,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,2.3563,20.0054,9.5984,-0.8754,-0.8593,9.7589,83.0428,-0.1329,10.5571,1,1,0.377244,1,10,2,4,2,2
2092,8.368,8368.0,-232.2956,-48.2545,-97.5986,-95.9738,104590.9622,97669.5777,104827.1457,118082.9259,134873.5849,61730.3210,62515.1571,45686.7798,0.4478,0.4543,0.4478,0.4543,4.1751,4.2422,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,1.8556,20.5387,9.5781,-0.9196,-0.9210,9.7055,83.0546,-0.0980,10.6803,1,1,0.377244,1,10,2,4,2,2
2093,8.372,8372.0,11.3660,116.5660,-46.0474,-259.2688,104597.9984,97663.8244,104855.0178,118088.4243,134890.1642,61734.3982,62557.0103,45735.0996,0.4495,0.4540,0.4495,0.4540,4.1719,4.2388,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,1.5291,20.9812,9.5533,-0.9602,-0.9840,9.6403,83.0612,-0.0737,10.7740,1,1,0.377244,1,10,2,4,2,2
2094,8.376,8376.0,NaN,NaN,NaN,NaN,104573.6908,97660.0857,104836.4631,118028.7253,134841.2561,61725.6198,62470.4387,45691.6384,0.4520,0.4528,0.4520,0.4528,4.1924,4.2251,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,1.2174,21.4363,9.5496,-0.9784,-1.0136,9.6195,83.0691,-0.0512,10.8664,1,1,0.377244,1,10,2,4,2,2
2095,8.380,8380.0,NaN,NaN,NaN,NaN,104555.8603,97660.5203,104796.6429,117994.4830,134801.3775,61719.1260,62385.6435,45618.9903,0.4528,0.4504,0.4528,0.4504,4.2050,4.2115,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,0.8098,22.0793,9.5648,-0.9577,-0.9683,9.6843,83.0826,-0.0231,10.9895,1,1,0.377244,1,10,2,4,2,2


✅ CLEANING APPLIED CORRECTLY — CELL 8 — Active-only filtering applied


In [11]:
# ============================================================
# CELL 9 — Validate labels after filtering
# This cell checks that non-T0 exported rows are active/action rows with task labels.
# ============================================================

checks = {}

checks["has_rows"] = len(df_labelonly) > 0
checks["has_expected_columns"] = set(target_order).issubset(set(df_labelonly.columns))
checks["no_extra_columns_remaining"] = len(set(df_labelonly.columns) - ALLOWED_SET) == 0

if is_task0:
    checks["t0_kept_all_rows"] = len(df_labelonly) == before_rows
else:
    checks["all_active_equals_1"] = bool((df_labelonly["active"].astype("int64") == 1).all())
    checks["all_label_action_equals_1"] = bool((df_labelonly["label_action"].astype("int64") == 1).all())
    checks["no_rest_label_11_in_active_export"] = bool((df_labelonly["label_11"].astype("int64") != 0).all())
    checks["task_target_matches_label_11"] = bool((df_labelonly["task_target"].astype("int64") == df_labelonly["label_11"].astype("int64")).all())

for k, v in checks.items():
    print(f"{k}: {v}")

passed = all(checks.values())
print_step_result("CELL 9 — Final label-only validation", passed)


has_rows: True
has_expected_columns: True
no_extra_columns_remaining: True
all_active_equals_1: True
all_label_action_equals_1: True
no_rest_label_11_in_active_export: True
task_target_matches_label_11: True
✅ CLEANING APPLIED CORRECTLY — CELL 9 — Final label-only validation


In [12]:
# ============================================================
# CELL 10 — Export Phase 2B labelonly CSV and audit logs
# This cell saves the active-only CSV and logs exactly what was changed.
# ============================================================

output_csv = OUTPUT_DIR / f"{INPUT_CSV.stem}_phase2B_labelonly_checked.csv"
schema_report_csv = OUTPUT_DIR / "phase2B_column_guard_schema_report_single.csv"
invalid_log_csv = OUTPUT_DIR / "phase2B_column_invalid_or_skipped_single.csv"
filter_log_csv = OUTPUT_DIR / "phase2B_active_zero_rows_removed_single.csv"
summary_json = OUTPUT_DIR / "phase2B_labelonly_summary_single.json"

df_labelonly.to_csv(output_csv, index=False)

schema_report = pd.DataFrame([{
    "file": str(INPUT_CSV),
    "raw_rows": len(df_raw),
    "raw_cols": df_raw.shape[1],
    "strict_original_guard_passed": strict_guard_passed,
    "missing_required_count": len(missing_required),
    "extra_count": len(extras),
    "extra_columns": ";".join(extras),
    "missing_required_columns": ";".join(missing_required),
    "schema_cols_after_canonical_keep": df_schema.shape[1],
    "columns_removed_in_colab_safe_repair": df_raw.shape[1] - df_schema.shape[1],
    "filter_mode": filter_mode,
    "rows_before_filter": before_rows,
    "rows_removed_active0": rows_removed,
    "rows_after_filter": len(df_labelonly),
    "all_active_after_filter": bool((df_labelonly["active"].astype(int) == 1).all()) if (not is_task0 and len(df_labelonly) > 0) else bool(is_task0),
}])
schema_report.to_csv(schema_report_csv, index=False)

pd.DataFrame([{
    "file": str(INPUT_CSV),
    "when": datetime.now().isoformat(timespec="seconds"),
    "missing": ";".join(missing_required),
    "extra": ";".join(extras),
    "note": "This records how the original strict Phase 2B guard evaluates the file."
}]).to_csv(invalid_log_csv, index=False)

pd.DataFrame([{
    "file": str(INPUT_CSV),
    "when": datetime.now().isoformat(timespec="seconds"),
    "rows_before": before_rows,
    "rows_removed": rows_removed,
    "rows_after": len(df_labelonly),
    "dest": str(output_csv),
}]).to_csv(filter_log_csv, index=False)

summary = schema_report.iloc[0].to_dict()
summary["output_csv"] = str(output_csv)
summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved labelonly CSV:", output_csv)
print("Saved schema report :", schema_report_csv)
print("Saved invalid log   :", invalid_log_csv)
print("Saved filter log    :", filter_log_csv)
print("Saved summary JSON  :", summary_json)

print("\nSaved CSV shape:", pd.read_csv(output_csv).shape)
display(pd.read_csv(output_csv).head())

passed = output_csv.exists() and schema_report_csv.exists() and filter_log_csv.exists()
print_step_result("CELL 10 — Phase 2B files exported", passed)


Saved labelonly CSV: /content/phase2B_debug_outputs/063_T24_HSMM_phase2B_labelonly_checked.csv
Saved schema report : /content/phase2B_debug_outputs/phase2B_column_guard_schema_report_single.csv
Saved invalid log   : /content/phase2B_debug_outputs/phase2B_column_invalid_or_skipped_single.csv
Saved filter log    : /content/phase2B_debug_outputs/phase2B_active_zero_rows_removed_single.csv
Saved summary JSON  : /content/phase2B_debug_outputs/phase2B_labelonly_summary_single.json

Saved CSV shape: (1097, 45)


,Timestamp_seconds,Timestamp_ms,EMG_Ch1,EMG_Ch2,EMG_Ch3,EMG_Ch4,EEG_Ch1,EEG_Ch2,EEG_Ch3,EEG_Ch4,EEG_Ch5,EEG_Ch6,EEG_Ch7,EEG_Ch8,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_PupilLeft,ET_PupilRight,ET_ValidityLeftEye,ET_ValidityRightEye,ET_Blink,ET_Fixation,ET_Worn,ET_DistanceLeft,ET_DistanceRight,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll,active_raw,active,active_prob,label_action,subject_id,task,trial,label_11,task_target
0,3.996,3996.0,-35.6730,7.5761,19.9850,108.1464,104412.1595,97503.4998,104708.9312,117952.2484,134692.5810,61608.7579,62264.5347,45589.4857,0.6024,0.6104,0.6024,0.6104,4.3070,4.3687,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-26.9252,2.1304,-12.4931,-0.5567,-4.5976,8.9107,61.3502,-42.3269,9.3288,1,1,0.377244,1,10,2,4,2,2
1,4.000,4000.0,78.8217,-149.2976,-220.4043,15.8415,104415.8378,97510.5967,104693.6469,117970.2568,134702.0429,61611.5667,62264.0619,45562.3994,0.5972,0.6103,0.5972,0.6103,4.3575,4.3737,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-26.4346,1.6322,-12.2411,-0.5483,-4.6155,8.8952,61.2693,-42.3602,9.3155,1,1,0.377244,1,10,2,4,2,2
2,4.004,4004.0,232.5416,-27.1953,-131.1567,-62.6811,104445.4114,97525.7896,104729.5683,118029.8235,134762.6485,61628.4776,62391.4939,45650.5991,0.5954,0.6110,0.5954,0.6110,4.3721,4.3964,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-25.9348,1.1527,-11.9698,-0.5526,-4.6271,8.8649,61.1919,-42.4009,9.3027,1,1,0.377244,1,10,2,4,2,2
3,4.008,4008.0,164.6698,65.0828,50.5351,-30.8734,104448.5902,97529.2024,104754.3236,118023.2275,134765.1745,61627.1919,62412.2693,45693.8504,0.5948,0.6100,0.5948,0.6100,4.3721,4.4520,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-25.3309,0.5826,-11.5988,-0.5776,-4.6320,8.8112,61.1000,-42.4330,9.2862,1,1,0.377244,1,10,2,4,2,2
4,4.012,4012.0,-95.5774,-39.0934,154.8412,82.2288,104420.3970,97515.3359,104725.5367,117970.0322,134710.8885,61607.8690,62302.1497,45616.2374,0.5938,0.6088,0.5938,0.6088,4.3819,4.4500,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-24.9359,0.1354,-11.3474,-0.5924,-4.6352,8.7764,61.0380,-42.4454,9.2732,1,1,0.377244,1,10,2,4,2,2


✅ CLEANING APPLIED CORRECTLY — CELL 10 — Phase 2B files exported


In [13]:
# ============================================================
# CELL 11 — Final Phase 2B summary
# This cell prints one final clear result for professor review.
# ============================================================

print("PHASE 2B FINAL SUMMARY")
print("=" * 70)
print("Input file:", INPUT_CSV.name)
print("Raw shape:", df_raw.shape)
print("Strict original guard passed:", strict_guard_passed)
print("Missing required columns:", missing_required if missing_required else "None")
print("Extra unsupported columns:", extras if extras else "None")
print("Canonical/repaired shape before active filter:", df_schema.shape)
print("Rows before filter:", before_rows)
print("Rows removed active==0:", rows_removed)
print("Rows after filter:", len(df_labelonly))
print("Final output columns:", len(df_labelonly.columns))
print("Output file:", output_csv)

final_passed = (
    output_csv.exists()
    and len(df_labelonly) > 0
    and len(set(df_labelonly.columns) - ALLOWED_SET) == 0
    and (is_task0 or (df_labelonly["active"].astype("int64") == 1).all())
)

print()
print_step_result("FINAL PHASE 2B VALIDATION", final_passed)


PHASE 2B FINAL SUMMARY
Input file: 063_T24_HSMM.csv
Raw shape: (2097, 58)
Strict original guard passed: False
Missing required columns: None
Extra unsupported columns: ['ET_Gaze3dEyeballXLeft', 'ET_Gaze3dEyeballXRight', 'ET_Gaze3dEyeballYLeft', 'ET_Gaze3dEyeballYRight', 'ET_Gaze3dEyeballZLeft', 'ET_Gaze3dEyeballZRight', 'ET_Gaze3dOpticalAxisXLeft', 'ET_Gaze3dOpticalAxisXRight', 'ET_Gaze3dOpticalAxisYLeft', 'ET_Gaze3dOpticalAxisYRight', 'ET_Gaze3dOpticalAxisZLeft', 'ET_Gaze3dOpticalAxisZRight', 'ET_TimeSignal']
Canonical/repaired shape before active filter: (2097, 45)
Rows before filter: 2097
Rows removed active==0: 1000
Rows after filter: 1097
Final output columns: 45
Output file: /content/phase2B_debug_outputs/063_T24_HSMM_phase2B_labelonly_checked.csv

✅ CLEANING APPLIED CORRECTLY — FINAL PHASE 2B VALIDATION
